In [1]:
import pandas as pd

In [1]:
!pwd

/storage/Arushi/070225/DGL_on_evoKG/DGL_onWholeEvoKG/Github/4-Ploting


# Validation

In [9]:
import os
import re
import pandas as pd

# Set path to your log files directory
log_dir = "../2-DGL_Training/Training_files/"

# Dictionary to store metrics
metrics_dict = {}

# Loop through all .log files in the directory
for filename in os.listdir(log_dir):
    if filename.endswith(".log"):
        model_name = filename.split("_")[0].lower()  # Extract model type
        filepath = os.path.join(log_dir, filename)
        with open(filepath, 'r') as file:
            for line in file:
                if "Valid average" in line:
                    match = re.search(r"Valid average (\w+@?\d*): ([\d.]+)", line)
                    if match:
                        metric, value = match.groups()
                        metric = metric.upper()
                        if model_name not in metrics_dict:
                            metrics_dict[model_name] = {}
                        metrics_dict[model_name][metric] = float(value)

# Convert to DataFrame
df = pd.DataFrame.from_dict(metrics_dict, orient="index")
df = df.rename(columns={"HITS@1": "Hit@1", "HITS@3": "Hit@3", "HITS@10": "Hit@10"})
df = df[["Hit@1", "Hit@3", "Hit@10", "MR", "MRR"]]  # Reorder columns
df.reset_index(inplace=True)
df.rename(columns={"index": "Model Type"}, inplace=True)



# Optionally save to CSV
df.to_csv("7_model_validation_metrics.csv", index=False)
df


,Model Type,Hit@1,Hit@3,Hit@10,MR,MRR
0,complex,0.724569,0.874440,0.980548,1.962164,0.812507
1,rescal,0.779446,0.896530,0.987536,1.753881,0.848793
2,rotate,0.762287,0.889526,0.982358,1.844754,0.837085
3,transr,0.829765,0.936423,0.995371,1.470378,0.888964
4,simple,0.748405,0.879537,0.981146,1.912511,0.826708
5,transe,0.007430,0.033001,0.217262,13.851715,0.095354
6,dismult,0.721259,0.860274,0.970516,2.121925,0.805246


# Test Metrric

In [7]:
import os
import re
import pandas as pd

# Directory containing your .log files
log_dir = "../2-DGL_Training/Training_files/"

# Dictionary to store test metrics
test_metrics = {}

# Parse each .log file
for filename in os.listdir(log_dir):
    if filename.endswith(".log"):
        model_name = filename.split("_")[0].lower()  # Normalize model name
        filepath = os.path.join(log_dir, filename)
        with open(filepath, 'r') as file:
            for line in file:
                if "Test average" in line:
                    match = re.search(r"Test average (\w+@?\d*): ([\d.]+)", line)
                    if match:
                        metric, value = match.groups()
                        metric = metric.upper()
                        if model_name not in test_metrics:
                            test_metrics[model_name] = {}
                        test_metrics[model_name][metric] = float(value)

# Convert to DataFrame
test_df = pd.DataFrame.from_dict(test_metrics, orient="index")
test_df = test_df.rename(columns={"HITS@1": "Hit@1", "HITS@3": "Hit@3", "HITS@10": "Hit@10"})
test_df = test_df[["Hit@1", "Hit@3", "Hit@10", "MR", "MRR"]]  # Order columns
test_df.reset_index(inplace=True)
test_df.rename(columns={"index": "Model Type"}, inplace=True)

# Display the final test metrics DataFrame
print(test_df)

# Optionally save to CSV
test_df.to_csv("7_model_test_metrics.csv", index=False)


  Model Type     Hit@1     Hit@3    Hit@10         MR       MRR
0    complex  0.730747  0.876897  0.980729   1.944389  0.816637
1     rescal  0.787950  0.904871  0.991125   1.680248  0.856278
2     rotate  0.795229  0.913277  0.987603   1.668392  0.862608
3     transr  0.849174  0.949283  0.996593   1.386618  0.903491
4     simple  0.742474  0.878621  0.980827   1.924839  0.823297
5     transe  0.006206  0.031862  0.200326  14.027409  0.093008
6    dismult  0.710838  0.860277  0.971560   2.121447  0.799925


# LossPlots

In [13]:
import os
import re
import pandas as pd

# Directory where your log files are stored
log_dir = "../2-DGL_Training/Training_files/"

# List of log files to include
target_logs = {
    "complex_64_emb_final_2.log": "Complex",
    "Simple_64_emb_final.log": "Simple",
    "dismult_64_emb_final_2.log": "Dismult",
    "RESCAL_64_emb_final.log": "Rescal",
    "rotatE_64_emb_final.log": "RotatE",
    "transE_64_emb_final.log": "TransE"
}

# Dictionary to collect model loss series
loss_data = {}

# Process each target log file
for log_filename, model_name in target_logs.items():
    filepath = os.path.join(log_dir, log_filename)

    steps = []
    losses = []

    with open(filepath, 'r') as file:
        for line in file:
            if "average loss" in line:
                match = re.search(r"\((\d+)/\d+\).*?average loss: ([\d.]+)", line)
                if match:
                    step, loss = match.groups()
                    steps.append(int(step))
                    losses.append(float(loss))

    # Store model loss with step as index
    loss_data[model_name + "_Avloss"] = pd.Series(data=losses, index=steps)

# Combine all models into one DataFrame (aligned by Step)
loss_df = pd.DataFrame(loss_data)
loss_df.index.name = "Step"
loss_df.reset_index(inplace=True)


# print(loss_df.head())

# save to CSV
loss_df.to_csv("filtered_stepwise_losses.csv", index=False)
loss_df

,Step,Complex_Avloss,Simple_Avloss,Dismult_Avloss,Rescal_Avloss,RotatE_Avloss,TransE_Avloss
0,10000,0.693150,0.693147,0.693148,0.693207,0.648749,0.818607
1,20000,0.692024,0.693147,0.689875,0.687065,0.640729,0.807274
2,30000,0.678654,0.693147,0.672317,0.645898,0.637635,0.807840
3,40000,0.634386,0.693147,0.626406,0.608062,0.629770,0.808369
4,50000,0.606169,0.692521,0.610698,0.592869,0.620710,0.808799
...,...,...,...,...,...,...,...
404,4050000,0.465632,0.486878,0.546226,0.371516,0.402738,0.810782
405,4060000,0.465475,0.486752,0.546210,0.371415,0.402469,0.810789
406,4070000,0.465492,0.486822,0.546131,0.371412,0.402012,0.810778
407,4080000,0.465321,0.486545,0.546060,0.371309,0.402104,0.810783
